In [ ]:
# 1. CatBoost Kütüphanesini Colab'e Kuruyoruz
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

import warnings
warnings.filterwarnings('ignore')

In [ ]:
print("--- 1. Veriler Yükleniyor ve Özellikler Üretiliyor ---")
train_df = pd.read_csv('/content/drive/MyDrive/Datathon/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/Datathon/test_x.csv')

# Feature Engineering (Özellik Mühendisliği)
def ozellikleri_uret(df):
    df_copy = df.copy()
    if 'rem_yuzdesi' in df_copy.columns and 'derin_uyku_yuzdesi' in df_copy.columns:
        df_copy['toplam_kaliteli_uyku_yuzdesi'] = df_copy['rem_yuzdesi'] + df_copy['derin_uyku_yuzdesi']
    if 'stres_skoru' in df_copy.columns and 'gunluk_calisma_saati' in df_copy.columns:
        df_copy['zihinsel_yuk_endeksi'] = df_copy['stres_skoru'] * df_copy['gunluk_calisma_saati']
    if 'gecelik_uyanma_sayisi' in df_copy.columns and 'uykuya_dalma_suresi_dk' in df_copy.columns:
        df_copy['uyku_zorlugu_endeksi'] = df_copy['gecelik_uyanma_sayisi'] * df_copy['uykuya_dalma_suresi_dk']
    if 'gunluk_adim_sayisi' in df_copy.columns and 'dinlenik_nabiz_bpm' in df_copy.columns:
        df_copy['fiziksel_dinclik_orani'] = df_copy['gunluk_adim_sayisi'] / (df_copy['dinlenik_nabiz_bpm'] + 0.1)
    return df_copy

train_df = ozellikleri_uret(train_df)
test_df = ozellikleri_uret(test_df)

# BÜTÜN SÜTUNLARI TUTUYORUZ (Sadece id ve hedefi ayırıyoruz)
hedef = 'bilissel_performans_skoru'
y_train_tam = train_df[hedef].copy()
X_train_tam = train_df.drop(columns=['id', hedef])
X_test_tam = test_df.drop(columns=['id'])

kategorik_sutunlar = X_train_tam.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
sayisal_sutunlar = X_train_tam.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()

--- 1. Veriler Yükleniyor ve Özellikler Üretiliyor ---


In [ ]:
print("--- 2. Kategorik Veriler Olasılıksal Dağılıma Göre Dolduruluyor ---")
def dagilima_gore_doldur(train_data, test_data, col):
    olasiliklar = train_data[col].value_counts(normalize=True)
    kategoriler = olasiliklar.index
    oranlar = olasiliklar.values

    train_bos = train_data[col].isnull()
    if train_bos.sum() > 0:
        train_data.loc[train_bos, col] = np.random.choice(kategoriler, size=train_bos.sum(), p=oranlar)

    test_bos = test_data[col].isnull()
    if test_bos.sum() > 0:
        test_data.loc[test_bos, col] = np.random.choice(kategoriler, size=test_bos.sum(), p=oranlar)
    return train_data, test_data

for cat_col in kategorik_sutunlar:
    X_train_tam, X_test_tam = dagilima_gore_doldur(X_train_tam, X_test_tam, cat_col)

In [ ]:
print("--- 3. Encoding (One-Hot) ---")
X_train_enc = pd.get_dummies(X_train_tam, columns=kategorik_sutunlar, drop_first=True)
X_test_enc = pd.get_dummies(X_test_tam, columns=kategorik_sutunlar, drop_first=True)
X_train_enc, X_test_enc = X_train_enc.align(X_test_enc, join='left', axis=1, fill_value=0)

In [ ]:
print("--- 4. Sayısal Veriler MICE İle Dolduruluyor ---")
imputer = IterativeImputer(random_state=42, max_iter=10)
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train_enc), columns=X_train_enc.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test_enc), columns=X_test_enc.columns)

In [ ]:
print("--- 5. Ölçeklendirme (Robust Scaling) ---")
scaler = RobustScaler()
X_train_final = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X_train_imp.columns)
X_test_final = pd.DataFrame(scaler.transform(X_test_imp), columns=X_test_imp.columns)

# Validation İçin Ayırma
X_tr, X_val, y_tr, y_val = train_test_split(X_train_final, y_train_tam, test_size=0.2, random_state=42)

print("\n🚀 VERİ HAZIR! ŞAMPİYONLUK STRATEJİSİ (TUNING + STACKING) BAŞLIYOR...")
print("Bu işlem 5-10 dakika sürebilir...\n")

In [ ]:
# 6. MODELLER VE ARAMA UZAYLARI
models_and_params = {
    "LightGBM": (
        LGBMRegressor(random_state=42, verbose=-1),
        {'n_estimators': [200, 500, 800], 'max_depth': [5, 10, -1], 'learning_rate': [0.01, 0.05, 0.1], 'subsample': [0.7, 0.8, 1.0], 'num_leaves': [31, 50, 100]}
    ),
    "XGBoost": (
        XGBRegressor(random_state=42, objective='reg:squarederror'),
        {'n_estimators': [200, 500, 800], 'max_depth': [3, 5, 7], 'learning_rate': [0.01, 0.05, 0.1], 'subsample': [0.7, 0.8, 1.0], 'colsample_bytree': [0.7, 0.8, 1.0]}
    ),
    "CatBoost": (
        CatBoostRegressor(random_state=42, verbose=0),
        {'iterations': [200, 500, 800], 'depth': [4, 6, 8], 'learning_rate': [0.01, 0.05, 0.1], 'l2_leaf_reg': [1, 3, 5]}
    )
}

best_estimators = {}

In [ ]:
# 7. HER BİR MODEL İÇİN TUNING
for name, (model, params) in models_and_params.items():
    print(f"⏳ {name} optimize ediliyor...")
    n_jobs_param = -1 if name != "CatBoost" else None

    search = RandomizedSearchCV(
        estimator=model, param_distributions=params, n_iter=15, cv=3,
        scoring='neg_root_mean_squared_error', random_state=42, n_jobs=n_jobs_param
    )
    search.fit(X_tr, y_tr)
    best_estimators[name] = search.best_estimator_

    y_pred_single = best_estimators[name].predict(X_val)
    print(f"✅ {name} Bitti! RMSE : {np.sqrt(mean_squared_error(y_val, y_pred_single)):.4f}\n")

In [ ]:
# 8. ENSEMBLE (STACKING) AŞAMASI
print("🌟 STACKING MODELİ KURULUYOR (Güçler Birleşiyor)...")
estimators = [
    ('lgb', best_estimators["LightGBM"]),
    ('xgb', best_estimators["XGBoost"]),
    ('cat', best_estimators["CatBoost"])
]

stack_model = StackingRegressor(estimators=estimators, final_estimator=Ridge(), cv=5, n_jobs=-1)
stack_model.fit(X_tr, y_tr)

In [ ]:
# 9. NİHAİ DEĞERLENDİRME
y_pred_stack = stack_model.predict(X_val)
final_rmse = np.sqrt(mean_squared_error(y_val, y_pred_stack))
final_r2 = r2_score(y_val, y_pred_stack)

print("*" * 50)
print(f"🏆 NİHAİ ŞAMPİYONLUK MODELİ (STACKING) SONUÇLARI")
print(f"📉 Validation RMSE : {final_rmse:.4f}")
print(f"📈 Validation R²   : {final_r2:.4f}")
print("*" * 50)

--- 2. Kategorik Veriler Olasılıksal Dağılıma Göre Dolduruluyor ---
--- 3. Encoding (One-Hot) ---
--- 4. Sayısal Veriler MICE İle Dolduruluyor ---
--- 5. Ölçeklendirme (Robust Scaling) ---

🚀 VERİ HAZIR! ŞAMPİYONLUK STRATEJİSİ (TUNING + STACKING) BAŞLIYOR...
Bu işlem 5-10 dakika sürebilir...

⏳ LightGBM optimize ediliyor...
✅ LightGBM Bitti! RMSE : 1.2362

⏳ XGBoost optimize ediliyor...
✅ XGBoost Bitti! RMSE : 1.2360

⏳ CatBoost optimize ediliyor...
✅ CatBoost Bitti! RMSE : 1.2319

🌟 STACKING MODELİ KURULUYOR (Güçler Birleşiyor)...
**************************************************
🏆 NİHAİ ŞAMPİYONLUK MODELİ (STACKING) SONUÇLARI
📉 Validation RMSE : 1.2306
📈 Validation R²   : 0.6984
**************************************************


In [ ]:
print("--- 🌟 ŞAMPİYON MODELLERİN EN İYİ HİPERPARAMETRELERİ 🌟 ---\n")

# Sadece bizim optimize ettiğimiz (aradığımız) parametreleri filtreleyip gösterelim
aranan_parametreler = {
    "LightGBM": ['n_estimators', 'max_depth', 'learning_rate', 'subsample', 'num_leaves'],
    "XGBoost": ['n_estimators', 'max_depth', 'learning_rate', 'subsample', 'colsample_bytree'],
    "CatBoost": ['iterations', 'depth', 'learning_rate', 'l2_leaf_reg']
}

for name, model in best_estimators.items():
    print(f"🏆 {name}:")
    tum_parametreler = model.get_params()
    for p in aranan_parametreler[name]:
        print(f"   {p}: {tum_parametreler.get(p)}")
    print("-" * 30)


print("\n--- 🚀 NİHAİ TAHMİNLER ALINIYOR ---")
# Eğittiğimiz Stacking modeline test setimizi (X_test_final) vererek tahmin yapıyoruz
y_test_pred = stack_model.predict(X_test_final)

--- 🌟 ŞAMPİYON MODELLERİN EN İYİ HİPERPARAMETRELERİ 🌟 ---

🏆 LightGBM:
   n_estimators: 800
   max_depth: 10
   learning_rate: 0.01
   subsample: 0.7
   num_leaves: 31
------------------------------
🏆 XGBoost:
   n_estimators: 800
   max_depth: 7
   learning_rate: 0.01
   subsample: 0.7
   colsample_bytree: 0.7
------------------------------
🏆 CatBoost:
   iterations: 800
   depth: 4
   learning_rate: 0.05
   l2_leaf_reg: 3
------------------------------

--- 🚀 NİHAİ TAHMİNLER ALINIYOR ---


In [ ]:
print("--- NİHAİ TAHMİNLER ALINIYOR ---")
# Eğittiğimiz Stacking modeline test setimizi (X_test_final) vererek tahmin yapıyoruz
y_test_pred = stack_model.predict(X_test_final)

# Yarışma formatına uygun bir DataFrame oluşturuyoruz
# 'id' sütununu orijinal test_df'den alıyoruz
submission = pd.DataFrame({
    'id': test_df['id'],
    'bilissel_performans_skoru': y_test_pred
})

# CSV olarak kaydetme (index=False diyerek satır numaralarının eklenmesini engelliyoruz)
submission.to_csv('submission.csv', index=False)

print("✅ 'submission.csv' dosyası başarıyla oluşturuldu!")
print("Dosyayı sol taraftaki klasör simgesine tıklayarak indirebilir ve platforma yükleyebilirsiniz.")
display(submission.head())

--- NİHAİ TAHMİNLER ALINIYOR ---
✅ 'submission.csv' dosyası başarıyla oluşturuldu!
Dosyayı sol taraftaki klasör simgesine tıklayarak indirebilir ve platforma yükleyebilirsiniz.


,id,bilissel_performans_skoru
0,1,6.029357
1,2,6.491042
2,3,3.113752
3,4,7.222942
4,5,3.670542
